In [0]:
%sql

DROP CONNECTION IF EXISTS youtube_earthquake_connection;

CREATE CONNECTION youtube_earthquake_connection
TYPE HTTP
OPTIONS (
    host = 'https://earthquake.usgs.gov',
    port = 443,
    base_path = '/earthquakes/feed/v1.0/',
    bearer_token = 'na'
)

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient();
conn = w.connections.get("youtube_earthquake_connection")
print(conn)
base_url = f"{conn.options['host']}{conn.options['base_path']}";

In [0]:
dbutils.widgets.text("catalog_name","youtube_dev","youtube_dev");
catalog_name = dbutils.widgets.get("catalog_name");
print(catalog_name)

In [0]:


spark.sql(f"use catalog {catalog_name}");

spark.sql("use schema bronze")

spark.sql("""
create volume if not exists earthquake_data_bronze
""");

In [0]:
import requests
import json
import datetime

current_date = datetime.datetime.now().strftime("%Y-%m-%d")
url = f"{base_url}summary/all_day.geojson"
response = requests.get(url)

if response.status_code != 200:
    raise Exception("Failed to retrieve data")
else:
    print("data Retrieved Successfully")
    earthquake_bronze_data = response.json()
    dbutils.fs.put(
    f"/Volumes/{catalog_name}/bronze/earthquake_data_bronze/earthquake_data.json_{current_date}",
    json.dumps(earthquake_bronze_data),
    overwrite=True,
    )